In [1]:
import sys
sys.path.append('../src')
sys.path.append('../data')
import pandas as pd
import numpy as np
import os
import pymc as pm

In [ ]:
import arviz as az
import xarray as xr
from simulation import simulate_from_posterior
# Load the trace
trace = az.from_netcdf("trace_results.nc")

# Show summary of the posterior distributions
summary = az.summary(trace)
summary


ModuleNotFoundError: No module named 'simulate_posterior'

In [8]:
posterior = trace.posterior

# Flatten and sample N parameter sets
N_sim = 100
samples = posterior.stack(draws=("chain", "draw")).transpose("draws", ...).isel(draws=slice(0, N_sim))

# Optional: set fixed test params
sens = 0.8
spec = 1.0

In [9]:
from seir_model import SEIRModel

trajectories = []

for i in range(N_sim):
    model = SEIRModel(
        beta=float(samples['beta'][i]),
        sigma=float(samples['sigma'][i]),
        p_sym=float(samples['p_sym'][i]),
        gamma=float(samples['gamma'][i]),
        hosp_rate=float(samples['hosp_rate'][i]),
        mu=float(samples['mu'][i]),
        p_test=float(samples['p_test'][i]),
        sens=sens,
        spec=spec,
        S0=999_000,
        E0=1000,
        I0=0,
        R0=0,
        stochastic=False
    )
    
    result = model.simulate(days=120)
    trajectories.append(result)


AttributeError: 'SEIRModel' object has no attribute 'simulate'